# MCI-GRU: Train â†’ Test metrics â†’ Backtest â†’ Download

Workflow:

1. **Setup** â€” detect Colab, clone repo, install deps, resolve root
2. **FRED key** â€” Colab Secrets or manual paste (required for regime features)
3. **Data** â€” market CSV via Drive mount, file upload, or local path
4. **Config** â€” experiment hyperparameters
5. **Train** â€” `run_experiment.py` (Hydra); outputs under `results/<name>/<timestamp>/`
6. **Inspect** â€” `training_summary.json`, `run_metadata.json`, `averaged_predictions/`
7. **Backtest** â€” `tests/backtest_sp500.py --auto_save`
8. **Package** â€” zip run directory; Colab `files.download`

> **Colab runtime:** Switch to GPU before running â€” *Runtime â†’ Change runtime type â†’ T4 GPU*.

**Label horizon:** `model.label_t` = what the model predicts (e.g. 21-day rank). `--label_t` in the backtest = return window for the simulation. They need not match.


## 1. Environment, clone, and install


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

# â”€â”€ Colab detection â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
try:
    from google.colab import files as colab_files  # type: ignore
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False

COLAB_REPO_DIR = Path("/content/MCI-GRU")
MANUAL_REPO_ROOT = None  # Local override: e.g. Path(r"C:\Users\you\MCI-GRU")

# â”€â”€ Clone / pull (Colab only) â€” runs BEFORE resolve_repo_root â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
REPO_URL = "https://github.com/magilliam27/MCI-GRU.git"
BRANCH   = "main"  # change to your branch if needed

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        print(f"Cloning {REPO_URL} @ {BRANCH} â€¦")
        subprocess.run(
            ["git", "clone", "-b", BRANCH, REPO_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
    else:
        print("Repo already cloned â€” pulling latest â€¦")
        subprocess.run(["git", "-C", str(COLAB_REPO_DIR), "fetch", "origin"], check=False)
        subprocess.run(["git", "-C", str(COLAB_REPO_DIR), "checkout", BRANCH], check=False)
        subprocess.run(["git", "-C", str(COLAB_REPO_DIR), "pull", "origin", BRANCH], check=False)
    print("Installing dependencies â€¦")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(COLAB_REPO_DIR / "requirements.txt")],
        check=True,
    )
else:
    print("Local environment â€” skipped clone/install.")

# â”€â”€ Resolve repo root â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def resolve_repo_root() -> Path:
    if MANUAL_REPO_ROOT is not None:
        return Path(MANUAL_REPO_ROOT).resolve()
    cwd = Path.cwd().resolve()
    for p in (cwd, cwd.parent, cwd.parent.parent):
        if (p / "run_experiment.py").is_file():
            return p
    if IN_COLAB and COLAB_REPO_DIR.is_dir():
        return COLAB_REPO_DIR.resolve()
    raise FileNotFoundError(
        "Cannot find run_experiment.py. Set MANUAL_REPO_ROOT or clone the repo on Colab."
    )

REPO_ROOT = resolve_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT =", REPO_ROOT)
print("IN_COLAB  =", IN_COLAB)


## 2. FRED API key (required for regime features)


In [ ]:
# Option A (recommended for Colab): add a Secret named FRED_API_KEY
#   Secrets panel â‰¡ â†’ ðŸ”‘ â†’ "Add new secret" â†’ name: FRED_API_KEY
# Option B: paste the key directly below.
MY_FRED_KEY = ""  # paste key here if not using Colab Secrets

if not os.environ.get("FRED_API_KEY"):
    if IN_COLAB:
        try:
            from google.colab import userdata  # type: ignore
            _secret = userdata.get("FRED_API_KEY")
            if _secret:
                os.environ["FRED_API_KEY"] = _secret
                print("FRED_API_KEY loaded from Colab Secrets.")
        except Exception:
            pass

    if not os.environ.get("FRED_API_KEY") and MY_FRED_KEY.strip():
        os.environ["FRED_API_KEY"] = MY_FRED_KEY.strip()
        print("FRED_API_KEY set from MY_FRED_KEY.")

if not os.environ.get("FRED_API_KEY"):
    raise ValueError(
        "FRED_API_KEY is not set. "
        "Add it as a Colab Secret (key: FRED_API_KEY) or paste it into MY_FRED_KEY above."
    )
print(f"FRED_API_KEY is set (length {len(os.environ['FRED_API_KEY'])}).")


## 2b. MLflow tracking

MLflow is already installed via `requirements.txt` (no extra pip install needed).

| `MLFLOW_TRACKING_URI` | Where runs are stored |
|---|---|
| `"mlruns"` *(default)* | Local store under the repo root — bundled in the zip download |
| `"https://dagshub.com/<user>/<repo>.mlflow"` | DagsHub remote — persists after the runtime ends |
| `"http://<host>:5000"` | Self-hosted MLflow server |

With the local default, the `mlruns/` directory is zipped in cell 8 so you can run `mlflow ui` after downloading.

In [ ]:
# ── MLflow tracking configuration ─────────────────────────────────────────
ENABLE_MLFLOW = True          # set False to skip all MLflow tracking

# Tracking store — choose one:
#   "mlruns"                                   local, bundled in zip (default)
#   "https://dagshub.com/<user>/<repo>.mlflow" remote, persists after runtime
#   "http://<host>:5000"                       self-hosted server
MLFLOW_TRACKING_URI = "mlruns"

# Optional: override the MLflow experiment name (defaults to EXPERIMENT_NAME)
MLFLOW_EXPERIMENT_NAME = None   # e.g. "colab-runs"

# ── Resolve absolute path / URI for subprocess calls ─────────────────────
if ENABLE_MLFLOW:
    if "://" in MLFLOW_TRACKING_URI:
        _mlflow_uri_abs = MLFLOW_TRACKING_URI          # already a remote URI
    else:
        _mlruns_dir = REPO_ROOT / MLFLOW_TRACKING_URI
        _mlruns_dir.mkdir(parents=True, exist_ok=True)
        _mlflow_uri_abs = str(_mlruns_dir)             # absolute local path
    print(f"MLflow enabled      : {ENABLE_MLFLOW}")
    print(f"MLflow tracking URI : {_mlflow_uri_abs}")
else:
    _mlflow_uri_abs = None
    print("MLflow disabled — set ENABLE_MLFLOW=True to activate.")

## 3. Market data CSV

Three ways to provide the market universe CSV:

| Option | How |
|--------|-----|
| **Drive mount** | Set `USE_GOOGLE_DRIVE = True` and `DRIVE_CSV_RELPATH` (recommended for large files) |
| **File upload** | Leave defaults â€” Colab prompts for upload on first run |
| **Local path** | Set `LOCAL_MARKET_CSV` to an absolute path (when running locally) |

Regime inputs are **not** read from a file: `features=with_momentum` with `include_global_regime=true` leaves `regime_inputs_csv` null, so `load_regime_inputs` pulls macro series from the FRED API.


In [ ]:
# â”€â”€ Configure one of the three options â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
USE_GOOGLE_DRIVE  = False   # True â†’ mount Drive; False â†’ upload or local path
DRIVE_CSV_RELPATH = "sp500_2019_universe_data_through_2026.csv"  # path inside MyDrive
LOCAL_MARKET_CSV  = None    # e.g. Path(r"C:\data\sp500.csv")  â€” local runs only

# â”€â”€ Resolution (no need to edit below) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DEFAULT_CSV_NAME = "sp500_2019_universe_data_through_2026.csv"
CSV_DIR = REPO_ROOT / "data" / "raw" / "market"
CSV_DIR.mkdir(parents=True, exist_ok=True)

if LOCAL_MARKET_CSV is not None:
    MARKET_CSV = Path(LOCAL_MARKET_CSV).resolve()
elif IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    MARKET_CSV = Path("/content/drive/MyDrive") / DRIVE_CSV_RELPATH
elif IN_COLAB:
    MARKET_CSV = CSV_DIR / DEFAULT_CSV_NAME
    if not MARKET_CSV.is_file():
        print("Upload your market universe CSV (saved as MARKET_CSV) â€¦")
        up = colab_files.upload()
        if not up:
            raise RuntimeError("Upload cancelled or empty.")
        MARKET_CSV.write_bytes(next(iter(up.values())))
        print(f"Saved to {MARKET_CSV}")
else:
    MARKET_CSV = CSV_DIR / DEFAULT_CSV_NAME

if not MARKET_CSV.is_file():
    raise FileNotFoundError(f"Market CSV not found: {MARKET_CSV}")
print("MARKET_CSV OK:", MARKET_CSV)


## 4. Experiment configuration

The `GRAPH_*` block at the top of the next cell controls the dynamic correlation graph and the new top-K / multi-feature edge upgrades. Defaults reproduce the legacy static graph; flip them on as documented in the cell to run the `correlation_dynamic_topk20_signed` (1b-lite, keeps strong negatives) or `correlation_dynamic_topk20_pos` (positive-only A/B sibling) experiments without touching the rest of the override list.


In [ ]:
# Mirrors Seed_test BASE: with_momentum + dynamic momentum + global regime.
EXPERIMENT_NAME = "notebook_train_backtest"
SEED        = 7
NUM_MODELS  = 10
HIS_T       = 60
LABEL_T     = 21

# ── Graph configuration ───────────────────────────────────────────────────
# Toggle the dynamic correlation graph + top-K / multi-feature edge upgrades
# (Lever 1a + 1c + 1b-lite). Defaults below match the static legacy graph so
# leaving everything as-is reproduces prior notebook behaviour byte-for-byte.
#
# To run the signed top-K experiment (correlation_dynamic_topk20_signed):
#   GRAPH_UPDATE_FREQUENCY_MONTHS = 6
#   GRAPH_TOP_K                   = 20
#   GRAPH_TOP_K_METRIC            = "abs_corr"  # keeps strong negatives
#   GRAPH_USE_MULTI_FEATURE_EDGES = True        # 4-d [corr, |c|, c^2, rank_pct]
#
# For the positive-only A/B sibling, switch to GRAPH_TOP_K_METRIC = "corr".
GRAPH_UPDATE_FREQUENCY_MONTHS = 0       # 0 = static graph (legacy default)
GRAPH_TOP_K                   = 0       # 0 = legacy threshold (judge_value)
GRAPH_TOP_K_METRIC            = "corr"  # "corr" or "abs_corr"; ignored if top_k==0
GRAPH_USE_MULTI_FEATURE_EDGES = False   # True => 4-d edge attr + GAT edge_dim=4

rel_market = MARKET_CSV.relative_to(REPO_ROOT).as_posix()

HYDRA_OVERRIDES = [
    f"experiment_name={EXPERIMENT_NAME}",
    f"seed={SEED}",
    # Use 'data=temporal_2019' (not '+data=') to override the default data config.
    "data=temporal_2019",
    "data.source=csv",
    f"data.filename={rel_market}",
    "features=with_momentum",
    "features.include_weekly_momentum=false",
    "features.momentum_encoding=binary",
    "features.momentum_blend_mode=dynamic",
    "features.momentum_dynamic_correction_fast_weight=0.15",
    "features.momentum_dynamic_rebound_fast_weight=0.7",
    "features.momentum_dynamic_lookback_periods=0",
    "features.momentum_dynamic_min_history=252",
    "features.momentum_dynamic_min_state_observations=3",
    "features.include_global_regime=true",
    "features.regime_enforce_lag_days=0",
    # Graph overrides — every flag has a legacy-safe default (see block above).
    f"graph.update_frequency_months={GRAPH_UPDATE_FREQUENCY_MONTHS}",
    f"graph.top_k={GRAPH_TOP_K}",
    f"graph.top_k_metric={GRAPH_TOP_K_METRIC}",
    f"graph.use_multi_feature_edges={str(GRAPH_USE_MULTI_FEATURE_EDGES).lower()}",
    f"model.his_t={HIS_T}",
    f"model.label_t={LABEL_T}",
    "model.use_multi_scale=false",
    "model.use_self_attention=false",
    "model.activation=elu",
    "model.latent_init_scale=0.02",
    "training.label_type=rank",
    f"training.num_models={NUM_MODELS}",
    "training.num_epochs=100",
    "training.early_stopping_patience=15",
]

# ── MLflow tracking overrides ─────────────────────────────────────────────
if ENABLE_MLFLOW and _mlflow_uri_abs:
    _mlflow_exp = MLFLOW_EXPERIMENT_NAME or EXPERIMENT_NAME
    HYDRA_OVERRIDES += [
        "tracking.enabled=true",
        f"tracking.tracking_uri={_mlflow_uri_abs}",
        f"tracking.experiment_name={_mlflow_exp}",
        "tracking.log_artifacts=true",
        "tracking.log_predictions=true",
    ]

print(f"Hydra overrides ({len(HYDRA_OVERRIDES)} args):")
for o in HYDRA_OVERRIDES:
    print("  ", o)

## 5. Train


In [ ]:
RUN_TRAIN = True


def run_training():
    cmd = [sys.executable, str(REPO_ROOT / "run_experiment.py"), *HYDRA_OVERRIDES]
    print("CMD:", " ".join(cmd[:3]), "... +", len(HYDRA_OVERRIDES), "overrides")
    proc = subprocess.run(
        cmd,
        cwd=str(REPO_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError(
            f"Training exit code {proc.returncode}. Full subprocess output printed above."
        )


if RUN_TRAIN:
    run_training()
else:
    print("Skipped training (RUN_TRAIN=False).")


## 6. Inspect latest run


In [ ]:
RESULTS_ROOT = REPO_ROOT / "results"


def is_hydra_timestamp_dir(name: str) -> bool:
    return bool(re.fullmatch(r"\d{8}_\d{6}", name))


def find_latest_run(results_root: Path, experiment_name: str) -> Path:
    base = results_root / experiment_name
    if not base.is_dir():
        raise FileNotFoundError(base)
    candidates = [p for p in base.iterdir() if p.is_dir() and is_hydra_timestamp_dir(p.name)]
    if not candidates:
        raise FileNotFoundError(f"No runs under {base}")
    return sorted(candidates)[-1]


LATEST_RUN = find_latest_run(RESULTS_ROOT, EXPERIMENT_NAME)
PRED_DIR   = LATEST_RUN / "averaged_predictions"

print("LATEST_RUN:", LATEST_RUN)
print("PRED_DIR:  ", PRED_DIR)

ts = LATEST_RUN / "training_summary.json"
if ts.is_file():
    with open(ts, encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2)[:6000])
else:
    print("No training_summary.json")

rm = LATEST_RUN / "run_metadata.json"
if rm.is_file():
    with open(rm, encoding="utf-8") as f:
        meta = json.load(f)
    print("run_metadata keys:", sorted(meta.keys()))


## 7. Backtest


In [ ]:
BACKTEST_LABEL_T = 5
TEST_START       = "2025-01-01"
TEST_END         = "2025-12-31"
TOP_K            = 10
BACKTEST_SUFFIX  = ""

rel_market_bt = MARKET_CSV.relative_to(REPO_ROOT).as_posix()

bt_cmd = [
    sys.executable,
    str(REPO_ROOT / "tests" / "backtest_sp500.py"),
    "--predictions_dir", str(PRED_DIR),
    "--data_file",       rel_market_bt,
    "--test_start",      TEST_START,
    "--test_end",        TEST_END,
    "--top_k",           str(TOP_K),
    "--label_t",         str(BACKTEST_LABEL_T),
    "--auto_save",
    "--plot",
]
if BACKTEST_SUFFIX:
    bt_cmd.extend(["--backtest_suffix", BACKTEST_SUFFIX])

# ── MLflow: link backtest as a child run under the training parent ─────────
# backtest_sp500.py auto-links via mlflow_run.json if it exists in LATEST_RUN.
if ENABLE_MLFLOW and _mlflow_uri_abs:
    bt_cmd.extend(["--enable_mlflow", "--mlflow_tracking_uri", _mlflow_uri_abs])

print("Backtest:", " ".join(bt_cmd))
proc = subprocess.run(
    bt_cmd,
    cwd=str(REPO_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    raise RuntimeError(
        f"Backtest exit code {proc.returncode}. Full subprocess output printed above."
    )

bt_name = "backtest" + (BACKTEST_SUFFIX or "")
BACKTEST_DIR = LATEST_RUN / bt_name
print("BACKTEST_DIR:", BACKTEST_DIR)
metrics_file = BACKTEST_DIR / "backtest_metrics.json"
if metrics_file.is_file():
    with open(metrics_file, encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2)[:4000])

In [ ]:
# ── MLflow run summary ────────────────────────────────────────────────────
from mci_gru.tracking.mlflow_manager import load_run_metadata

mlflow_meta = load_run_metadata(LATEST_RUN)
if mlflow_meta:
    print("MLflow run metadata:")
    print(json.dumps(mlflow_meta, indent=2))
    tracking_uri = mlflow_meta.get("tracking_uri", "")
    print()
    if "://" not in tracking_uri:
        # Local store — show command to open UI after downloading
        print("To view runs locally after downloading the zip:")
        print(f"  mlflow ui --backend-store-uri {tracking_uri}")
    else:
        print(f"Open {tracking_uri} in your browser to view runs.")
else:
    print("No MLflow metadata found (ENABLE_MLFLOW may be False, or training hasn't run yet).")

## 8. Zip and download


In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_base = REPO_ROOT / "results" / f"{EXPERIMENT_NAME}_bundle_{stamp}"
archive_path = shutil.make_archive(
    str(zip_base), "zip",
    root_dir=str(LATEST_RUN.parent),
    base_dir=LATEST_RUN.name,
)
print("Run archive:", archive_path)

if IN_COLAB and colab_files is not None:
    colab_files.download(archive_path)
else:
    print("Local: copy results from:", LATEST_RUN)

# ── Bundle the local MLflow store so runs are viewable after download ─────
if (
    ENABLE_MLFLOW
    and _mlflow_uri_abs is not None
    and "://" not in _mlflow_uri_abs
):
    _mlruns_path = Path(_mlflow_uri_abs)
    if _mlruns_path.is_dir():
        mlruns_zip_base = REPO_ROOT / "results" / f"mlruns_{stamp}"
        mlruns_archive = shutil.make_archive(
            str(mlruns_zip_base), "zip",
            root_dir=str(_mlruns_path.parent),
            base_dir=_mlruns_path.name,
        )
        print("MLflow archive:", mlruns_archive)
        print("After downloading, run: mlflow ui --backend-store-uri ./mlruns")
        if IN_COLAB and colab_files is not None:
            colab_files.download(mlruns_archive)